# Speech Processing

# Assignment 6

# Name: Saran Jayakumar Palani
# Reg No: BL.EN.U4AIE23131

## Objective
1. Estimate and visualize pitch, harmonics, and formants using FFT.
2. Estimate and visualize pitch contour and formants over time using STFT-based framing.
3. Compare visual distinguishability of pitch, harmonics, and formants in FFT and STFT views.
4. Plot narrowband and wideband spectrograms and infer the effect of window size.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import stft
from scipy.linalg import toeplitz

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
fs, x = wavfile.read('test.wav')
x = x.astype(np.float64)
if x.ndim > 1:
    x = np.mean(x, axis=1)
x = x / (np.max(np.abs(x)) + 1e-12)
t = np.arange(len(x)) / fs

plt.figure(figsize=(12, 3))
plt.plot(t, x)
plt.title('Speech Signal (test.wav)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.xlim(0, t[-1])
plt.show()

In [ ]:
def estimate_pitch_autocorr(frame, fs, fmin=60, fmax=350):
    frame = frame - np.mean(frame)
    if np.max(np.abs(frame)) < 1e-5:
        return np.nan
    r = np.correlate(frame, frame, mode='full')
    r = r[len(r) // 2:]
    lag_min = int(fs / fmax)
    lag_max = int(fs / fmin)
    if lag_max >= len(r):
        return np.nan
    seg = r[lag_min:lag_max + 1]
    if len(seg) == 0:
        return np.nan
    lag = np.argmax(seg) + lag_min
    if lag <= 0:
        return np.nan
    return fs / lag


def lpc_formants(frame, fs, order=14):
    frame = frame - np.mean(frame)
    frame = frame * np.hamming(len(frame))
    r = np.correlate(frame, frame, mode='full')
    r = r[len(r) // 2:]
    if len(r) < order + 1 or r[0] <= 0:
        return np.array([])
    R = toeplitz(r[:order])
    rhs = -r[1:order + 1]
    try:
        a = np.linalg.solve(R + 1e-8 * np.eye(order), rhs)
    except np.linalg.LinAlgError:
        return np.array([])
    A = np.concatenate(([1.0], a))
    roots = np.roots(A)
    roots = roots[np.imag(roots) >= 0.01]
    if len(roots) == 0:
        return np.array([])
    ang = np.arctan2(np.imag(roots), np.real(roots))
    freqs = np.sort(ang * fs / (2 * np.pi))
    bw = -0.5 * fs / (2 * np.pi) * np.log(np.abs(roots))
    valid = (freqs > 90) & (freqs < 5000) & (bw < 500)
    return freqs[valid]

In [ ]:
frame_dur = 0.04
N = int(frame_dur * fs)
center = len(x) // 2
start = max(0, center - N // 2)
frame = x[start:start + N]
if len(frame) < N:
    frame = np.pad(frame, (0, N - len(frame)))
frame = frame * np.hamming(len(frame))

nfft = 4096
X = np.fft.rfft(frame, n=nfft)
f = np.fft.rfftfreq(nfft, d=1 / fs)
mag = 20 * np.log10(np.abs(X) + 1e-8)

pitch0 = estimate_pitch_autocorr(frame, fs)
harmonics = np.array([])
if np.isfinite(pitch0) and pitch0 > 0:
    h = np.arange(1, int((f[-1] // pitch0)) + 1)
    harmonics = h * pitch0

formants_fft = lpc_formants(frame, fs, order=14)[:4]

plt.figure(figsize=(12, 5))
plt.plot(f, mag, linewidth=1.2, label='FFT magnitude')
if len(harmonics) > 0:
    plt.vlines(harmonics, ymin=np.min(mag), ymax=np.max(mag), colors='tab:green', alpha=0.25, linewidth=1.2, label='Harmonics')
if np.isfinite(pitch0):
    plt.axvline(pitch0, color='tab:red', linestyle='--', linewidth=1.8, label='Pitch')
for i, ff in enumerate(formants_fft):
    if i == 0:
        plt.axvline(ff, color='tab:orange', linestyle='-.', linewidth=1.8, label='Formants')
    else:
        plt.axvline(ff, color='tab:orange', linestyle='-.', linewidth=1.8)
plt.title('FFT: Pitch, Harmonics, and Formants')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB)')
plt.xlim(0, 5000)
plt.legend()
plt.show()

In [ ]:
win = int(0.03 * fs)
hop = int(0.01 * fs)
num_frames = 1 + (len(x) - win) // hop if len(x) >= win else 0
times = []
pitch_track = []
f1 = []
f2 = []
f3 = []

for i in range(num_frames):
    s = i * hop
    fr = x[s:s + win]
    t_mid = (s + win / 2) / fs
    times.append(t_mid)

    p = estimate_pitch_autocorr(fr, fs)
    if np.isfinite(p) and 60 <= p <= 350:
        pitch_track.append(p)
    else:
        pitch_track.append(np.nan)

    ff = lpc_formants(fr, fs, order=14)
    f1.append(ff[0] if len(ff) > 0 else np.nan)
    f2.append(ff[1] if len(ff) > 1 else np.nan)
    f3.append(ff[2] if len(ff) > 2 else np.nan)

times = np.array(times)
pitch_track = np.array(pitch_track)
f1 = np.array(f1)
f2 = np.array(f2)
f3 = np.array(f3)

f_stft, t_stft, Z = stft(x, fs=fs, window='hann', nperseg=1024, noverlap=768, nfft=2048, boundary=None)
S_db = 20 * np.log10(np.abs(Z) + 1e-8)

plt.figure(figsize=(12, 6))
plt.pcolormesh(t_stft, f_stft, S_db, shading='gouraud', cmap='magma')
plt.plot(times, pitch_track, color='cyan', linewidth=2, label='Pitch contour')
plt.plot(times, f1, color='lime', linewidth=1.5, label='F1')
plt.plot(times, f2, color='yellow', linewidth=1.5, label='F2')
plt.plot(times, f3, color='white', linewidth=1.5, label='F3')
plt.ylim(0, 5000)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('STFT Spectrogram with Pitch Contour and Formants')
plt.colorbar(label='Magnitude (dB)')
plt.legend(loc='upper right')
plt.show()

In [ ]:
f_n, t_n, Z_n = stft(x, fs=fs, window='hann', nperseg=1024, noverlap=900, nfft=2048, boundary=None)
f_w, t_w, Z_w = stft(x, fs=fs, window='hann', nperseg=256, noverlap=200, nfft=1024, boundary=None)

S_n = 20 * np.log10(np.abs(Z_n) + 1e-8)
S_w = 20 * np.log10(np.abs(Z_w) + 1e-8)

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
pcm1 = ax[0].pcolormesh(t_n, f_n, S_n, shading='gouraud', cmap='viridis')
ax[0].set_title('Narrowband Spectrogram (Longer Window)')
ax[0].set_ylabel('Frequency (Hz)')
ax[0].set_ylim(0, 5000)
fig.colorbar(pcm1, ax=ax[0], label='Magnitude (dB)')

pcm2 = ax[1].pcolormesh(t_w, f_w, S_w, shading='gouraud', cmap='viridis')
ax[1].set_title('Wideband Spectrogram (Shorter Window)')
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Frequency (Hz)')
ax[1].set_ylim(0, 5000)
fig.colorbar(pcm2, ax=ax[1], label='Magnitude (dB)')

plt.tight_layout()
plt.show()

## Analysis and Inference
- In the FFT magnitude plot, pitch appears as the fundamental frequency, harmonics appear as regularly spaced peaks at integer multiples, and formants appear as broader resonance regions in the spectral envelope.
- In STFT, the pitch contour is visible as a low-frequency track over time, while formants appear as relatively smooth bands (F1, F2, F3) with slower variation than harmonic fine structure.
- Pitch and harmonics are more directly distinguishable in high-resolution FFT of a short voiced segment, while temporal evolution is clearer in STFT.
- Narrowband spectrogram (longer window) improves frequency resolution and highlights harmonic lines, but reduces temporal sharpness.
- Wideband spectrogram (shorter window) improves temporal resolution and reveals rapid changes and pitch pulses better, but harmonics are less sharply separated.
- This confirms the standard time-frequency tradeoff: increasing window length improves frequency detail and decreasing window length improves time detail.